In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Button, TextBox, Cursor
from scipy.signal import butter, filtfilt

class InteractiveAEPickerSingle:
    def __init__( # Initialize data and picker
        self,
        data,
        fs,
        channel_ids=None,
        channels_names=None,
        initial_picks=None,
        save_dir="./saved_plots"
    ):
        self.data = np.atleast_2d(data)
        self.fs = fs
        self.n_channels, self.n_samples = self.data.shape
        self.time = np.arange(self.n_samples) / self.fs

        self.channel_ids = (
            channel_ids
            if channel_ids is not None
            else list(range(self.n_channels))
        )
        self.channels_names = (
            channels_names
            if channels_names is not None
            else [f"Ch {cid}" for cid in self.channel_ids]
        )

        self.current_ch_idx = 0
        self.picks = {} 

        # Variables to memorize the current zoom and keep the proportions constant
        self.current_xlim = None
        self.current_ylim = None
        self.cursor = None

        # Save directory configuration
        self.save_dir = save_dir
        if self.save_dir and not os.path.exists(self.save_dir):
            os.makedirs(self.save_dir, exist_ok=True)

        # Load pre-existing picks if available
        if initial_picks:
            self._load_initial_picks(initial_picks)

        self._setup_plot()
        self._connect_events()
        self._update_plot()

    def _load_initial_picks(self, initial_picks):
        for p in initial_picks:
            try:
                ch, t_val = None, None
                if isinstance(p, dict):
                    ch = p.get("ch", p.get("channel"))
                    t_val = p.get("time")
                elif isinstance(p, (list, tuple, np.ndarray)) and len(p) >= 2:
                    ch = p[0]
                    t_val = p[1]
                
                if ch is not None and ch in self.channel_ids:
                    self.picks[ch] = t_val
            except Exception:
                continue

    def apply_bandpass_filter(self, raw_time, amp_vals, low_cutoff_hz=0.0, high_cutoff_hz=1e6): # Applies the selected frequency filter
        if len(raw_time) < 4:
            return amp_vals

        dt = np.mean(np.diff(raw_time))
        if dt <= 0:
            return amp_vals

        dt_sec = dt if dt < 1e-3 else dt * 1e-6
        fs = 1.0 / dt_sec
        nyquist = 0.5 * fs

        if high_cutoff_hz >= nyquist:
            high_cutoff_hz = 0.95 * nyquist

        use_hp = low_cutoff_hz > 0
        use_lp = high_cutoff_hz > low_cutoff_hz

        try:
            if use_hp and use_lp:
                low_norm = low_cutoff_hz / nyquist
                high_norm = high_cutoff_hz / nyquist
                if low_norm <= 0 or high_norm >= 1 or low_norm >= high_norm:
                    return amp_vals
                b, a = butter(4, [low_norm, high_norm], btype="bandpass", analog=False)
            elif use_hp:
                low_norm = low_cutoff_hz / nyquist
                if low_norm <= 0 or low_norm >= 1:
                    return amp_vals
                b, a = butter(4, low_norm, btype="highpass", analog=False)
            elif use_lp:
                high_norm = high_cutoff_hz / nyquist
                if high_norm <= 0 or high_norm >= 1:
                    return amp_vals
                b, a = butter(4, high_norm, btype="lowpass", analog=False)
            else:
                return amp_vals

            return filtfilt(b, a, amp_vals)
        except Exception:
            return amp_vals

    def _setup_plot(self): # Creates interface and buttons
        self.fig, self.ax = plt.subplots(figsize=(12, 6.5))
        self.fig.subplots_adjust(bottom=0.28)  # Space for controls

        # UI controls
        ax_prev = plt.axes([0.05, 0.08, 0.08, 0.06])
        ax_next = plt.axes([0.14, 0.08, 0.08, 0.06])
        
        # Filter
        ax_box_hp = plt.axes([0.27, 0.08, 0.07, 0.06])
        ax_box_lp = plt.axes([0.39, 0.08, 0.07, 0.06])
        ax_filter = plt.axes([0.47, 0.08, 0.10, 0.06])

        # Buttons
        ax_print = plt.axes([0.59, 0.08, 0.12, 0.06])
        ax_save = plt.axes([0.73, 0.08, 0.22, 0.06])

        self.btn_prev = Button(ax_prev, '◀ Prev')
        self.btn_next = Button(ax_next, 'Next ▶')
        
        self.txt_hp = TextBox(ax_box_hp, 'HP: ', initial='0.0')
        self.txt_lp = TextBox(ax_box_lp, 'LP: ', initial='500.0')
        self.btn_filter = Button(ax_filter, 'Filter', color='#f39c12')

        self.btn_print = Button(ax_print, 'Print Picks', color='#2ecc71')
        self.btn_save = Button(ax_save, 'Save Plot (PNG)', color='#3498db')

        self.btn_prev.on_clicked(self.prev_channel)
        self.btn_next.on_clicked(self.next_channel)
        self.btn_filter.on_clicked(self.apply_filter_event)
        self.btn_print.on_clicked(self.print_picks_button)
        self.btn_save.on_clicked(self.save_plot_callback)

    def save_plot_callback(self, event): # Saves a PNG screenshot
        """Save current plot on drive."""
        cname = self.channels_names[self.current_ch_idx]
        cid = self.channel_ids[self.current_ch_idx]
        
        # Fallback di sicurezza se save_dir dovesse mancare
        save_directory = getattr(self, 'save_dir', './saved_plots')
        if not os.path.exists(save_directory):
            os.makedirs(save_directory, exist_ok=True)
            
        safe_cname = str(cname).replace(" ", "_").replace("/", "_")
        filename = f"channel_{cid}_{safe_cname}_pick.png"
        filepath = os.path.join(save_directory, filename)
        
        self.fig.savefig(filepath, format='png', dpi=300, bbox_inches='tight')
        print(f" Plot successfully saved in: {filepath}")

    def _connect_events(self):
        self.cid_press = self.fig.canvas.mpl_connect("button_press_event", self.on_press)
        self.cid_key = self.fig.canvas.mpl_connect("key_press_event", self.on_key)

    def _update_plot(self): # Displays the selected channel and its waveform
        if hasattr(self, 'ax') and len(self.ax.lines) > 0:
            try:
                self.current_xlim = self.ax.get_xlim()
                self.current_ylim = self.ax.get_ylim()
            except Exception:
                pass

        self.ax.clear()

        cid = self.channel_ids[self.current_ch_idx]
        cname = self.channels_names[self.current_ch_idx]
        raw_waveform = self.data[self.current_ch_idx]

        try:
            hp_khz = float(self.txt_hp.text)
            lp_khz = float(self.txt_lp.text)
            hp_hz = hp_khz * 1e3
            lp_hz = lp_khz * 1e3
        except ValueError:
            hp_hz, lp_hz = 0.0, 1e6

        waveform = self.apply_bandpass_filter(self.time, raw_waveform, low_cutoff_hz=hp_hz, high_cutoff_hz=lp_hz)

        self.ax.plot(
            self.time * 1e6,
            waveform,
            lw=1.0,
            color="#1f77b4",
            label="Filtered Signal"
        )

        if cid in self.picks and self.picks[cid] is not None:
            t_pick_us = self.picks[cid]
            self.ax.axvline(x=t_pick_us, color="red", linestyle="--", linewidth=1.8)
            self.ax.plot(t_pick_us, 0, "rx", markersize=8, markeredgewidth=2)

        if self.current_xlim is not None and self.current_ylim is not None:
            self.ax.set_xlim(self.current_xlim)
            self.ax.set_ylim(self.current_ylim)

        self.cursor = Cursor(self.ax, useblit=True, color='gray', linewidth=0.8, linestyle='--')

        self.ax.set_xlabel("Time [µs]")
        self.ax.set_ylabel("Amplitude [V]")
        self.ax.set_title(
            f"Channel: {cname} (ID: {cid}) | Channel {self.current_ch_idx + 1} of {self.n_channels} "
            f"(Filter: {hp_khz} - {lp_khz} kHz)\n"
            f"[Click SX: Add/Move Pick | Click DX: Delete | 'N'/'P': Change Channel]",
            fontsize=10,
            fontweight="bold",
        )
        self.ax.grid(True, linestyle=":", alpha=0.6)
        self.fig.canvas.draw_idle()

    def next_channel(self, event=None): # Navigates between sensors
        if self.current_ch_idx < self.n_channels - 1:
            self.current_ch_idx += 1
            self._update_plot()

    def prev_channel(self, event=None):
        if self.current_ch_idx > 0:
            self.current_ch_idx -= 1
            self._update_plot()

    def apply_filter_event(self, event):
        self._update_plot()

    def on_press(self, event): # Adds, moves, or deletes an arrival pick
        if event.inaxes != self.ax or event.xdata is None:
            return
        if self.fig.canvas.toolbar and self.fig.canvas.toolbar.mode != "":
            return

        cid = self.channel_ids[self.current_ch_idx]
        x_click_us = event.xdata

        if event.button == 1:
            self.picks[cid] = x_click_us
            self._update_plot()
        elif event.button == 3:
            if cid in self.picks:
                self.picks[cid] = None
                self._update_plot()

    def on_key(self, event): # Handles keyboard shortcuts
        if event.key == "n" or event.key == "right":
            self.next_channel()
        elif event.key == "p" or event.key == "left":
            self.prev_channel()
        elif event.key == "s":
            self.print_picks_to_console()

    def get_picks(self): # Collects the selected arrival times
        picks_list = []
        for cid in self.channel_ids:
            pick_val = self.picks.get(cid)
            if pick_val is not None:
                picks_list.append({
                    'time': round(float(pick_val), 8)/1e6,
                    'ch': int(cid)
                })
        return picks_list

    def print_picks_to_console(self): # Prints the results
        picks_list = self.get_picks()
        print("\n--- Final Picks List ---")
        print(picks_list)
        return picks_list

    def print_picks_button(self, event):
        self.print_picks_to_console()


def plot_single_channel_picking(data, fs, channel_ids=None, initial_picks=None):
    output_dir = '' #Directory where the figure will be saved

    picker = InteractiveAEPickerSingle(
        data=data,
        fs=fs,
        channel_ids=channel_ids,
        initial_picks=initial_picks,
        save_dir=output_dir
    )

    plt.show()
    return picker


# Use example
my_channels = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28] # List of channels to use
picker = plot_single_channel_picking(
    data=data_chunk[my_channels, :],
    fs=fsAE,
    channel_ids=my_channels,
    initial_picks=raw_picks,
)
evento_dict = picker.print_picks_to_console()